# ATHLLM-4B — Architecture, Distillation, RL & Kaggle Lab

Detailed research notebook for the Spark-X2.5-inspired ATHLLM foundation: architecture validation, parameter/memory analysis, teacher distillation, data quality, three RL tracks, quantization, ablations, and Kaggle continuation.

This is a research lab, not a fake one-click frontier training script. Kaggle is used for constrained experiments and checkpoint continuation; large teacher inference is performed offline.


## 1. Architecture hypothesis

```text
Token IDs → Embedding → 36 Decoder Blocks → RMSNorm → LM Head
                         │
                         ├─ S: Sliding Window
                         ├─ S: Sliding Window
                         ├─ S: Sliding Window
                         └─ F: Full Attention
                             repeat 3:1
```

Target configuration: hidden 2560, 36 layers, 16 query heads, 4 KV heads, head dimension 160, gated MLP 10240, 1M context target. The research question is whether this gives a better capability/compute ratio after distillation and RL.


In [ ]:
!git clone -q https://github.com/D-engahmed/ATHLLM.git || true
%cd ATHLLM
!git checkout feat/spark-x25-frontier-rebuild
!pip install -q -r requirements.txt


In [ ]:
import torch,yaml,math,json,subprocess
from pathlib import Path
cfg=yaml.safe_load(Path('configs/distillation_4b_kaggle.yaml').read_text())
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
 p=torch.cuda.get_device_properties(0); print('GPU:',p.name,'VRAM GB:',round(p.total_memory/1024**3,2))


## 2. Parameter anatomy and GQA

16 Q heads share 4 KV heads. The 4× sharing reduces KV-cache storage relative to standard MHA while preserving multiple query subspaces. The gated MLP uses gate/up/down projections. RMSNorm is applied before attention and MLP blocks.


In [ ]:
H,L,Q,KV,D,I,V=2560,36,16,4,160,10240,131072
print({'head_dim':D,'q_to_kv_ratio':Q/KV,'embedding_params_M':round(V*H/1e6,1)})
print('Attention/MLP are approximate because implementation details affect exact parameter count.')


## 3. Attention mechanics

```text
x → RMSNorm → Q(16h), K(4h), V(4h)
                 ↓
             GQA repeat KV
                 ↓
        causal + local/full mask
                 ↓
            attention
                 ↓
             O projection
                 ↓
              residual
                 ↓
              RMSNorm
                 ↓
             gated MLP
                 ↓
              residual
```

The repository implementation is a correctness reference. Before serious training, use optimized SDPA/FlashAttention-compatible kernels. A naive O(n²) full-attention implementation is not a practical 1M-token training solution.


In [ ]:
from athllm.models.spark25 import Spark25Config,ATHLLMSpark25
c=Spark25Config(vocab_size=4096,hidden_size=256,intermediate_size=1024,num_hidden_layers=4,num_attention_heads=8,num_key_value_heads=2)
m=ATHLLMSpark25(c); x=torch.randint(0,c.vocab_size,(2,64)); y=m(x)
print('input',tuple(x.shape),'logits',tuple(y.shape),'params',sum(p.numel() for p in m.parameters()))


## 4. Long-context compute reality

1M context is a capability target, not a Kaggle training length. Full attention scales quadratically in token count; sliding windows constrain local layers. Use a curriculum such as 2K → 4K → 8K → 16K → 32K → 64K → 128K before specialized long-context runs.


In [ ]:
def pairs(n,w=None): return n*n if w is None else n*min(n,w)
for n in [2048,4096,8192,16384,32768,65536]: print(n,'full_M=',round(pairs(n)/1e6,1),'window512_M=',round(pairs(n,512)/1e6,1))


## 5. Teacher → student distillation

Do not load a 397B frontier teacher into a normal Kaggle runtime. Use it offline to produce hard, high-value examples. Concentrate the expensive teacher on the hardest 5–10%; use cheap filters and deterministic verification elsewhere.

`candidate tasks → hard-task selection → teacher trajectories → independent verification → accepted dataset → ATHLLM-4B`


In [ ]:
funnel={'candidate_tasks':10_000_000,'hard_candidates':1_000_000,'verified_examples':500_000}
print(json.dumps(funnel,indent=2))


## 6. Data quality gates

Every sample should track provenance/license, language/domain, exact+near dedup status, semantic quality, spam/boilerplate checks, code execution results where applicable, contamination status, and split assignment. Start at 50B training tokens; scale to 100B/300B only after positive scaling evidence. 5T remains a long-term capacity target.


In [ ]:
!python -m athllm.data.build_manifest --help


## 7. Distillation objective

A practical objective can combine supervised response loss, optional logit-level KD when usable, and verified reasoning/coding objectives. Do not assume raw teacher chain-of-thought should be copied verbatim; prefer verified answers, structured solution artifacts, tool traces, and outcome-based supervision when appropriate.


## 8. Three RL tracks

**RL-1 Verifiable reasoning:** exact/symbolic/reproducible rewards for math, logic, science and planning.

**RL-2 Coding/agent:** repository repair, terminal use, tool calls and multi-step coding rewarded by actual tests/task completion.

**RL-3 Self-improvement:** task generation → environment → rollout → independent verification → failure mining → curriculum update → RL. Model-generated examples are never accepted solely because an LLM judged them good.


In [ ]:
from athllm.training.rl import RL_TRACKS
print(list(RL_TRACKS))


## 9. Kaggle experiment ladder

A. Tiny architecture smoke test → B. small SFT/distillation → C. short verifier-based RL → D. calibrated INT4 → E. held-out regression → F. checkpoint continuation. Persist checkpoints outside the ephemeral session.


## 10. Architecture ablations

Compare at approximately equal parameters/compute: SSSF baseline, SSSS local-only, FFFF small-context control, alternate global spacing (SSFS), and the future **Token Multi-Access** hypothesis. Measure perplexity, reasoning, coding, retrieval, long-context and agent success. Do not add the new architecture before the baseline is reproducible.


## 11. Token Multi-Access research hook

```text
token
 ├─ local context
 ├─ periodic global context
 ├─ compressed memory
 ├─ reasoning state
 └─ tool/environment state
        ↓ learned router
   next representation
```

This is the candidate ATHLLM research contribution. Its value must be established with controlled ablations, not assumed.


## 12. Evaluation contract

Pin checkpoint/model version, tokenizer, prompt, context length, decoding, tool budget, benchmark/evaluator versions and hardware. Report both gains and regressions. The objective is measured capability improvement, not a predetermined claim of universal frontier superiority.


In [ ]:
commit=subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
print(json.dumps({'git_commit':commit,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},indent=2))


## 13. Next engineering milestones

1. Optimized attention kernels.
2. Exact Spark checkpoint tensor mapping and conversion tests.
3. Streaming/packed-sequence data loader.
4. Teacher-generation adapters.
5. Independent math/code verifiers.
6. Real SFT/distillation trainer.
7. Three RL trainers.
8. Automated benchmark regression reports.
9. Calibrated INT4 export.
10. Larger scaling only after ablation evidence.
